# Brain Tumour Classification with Explainable AI (XAI)
**Single-regime · Stratified 80/10/10 · No augmentation · Proposed custom CNN (scratch) · VGG16 · VGG19 · ResNet50 · Fine-tuning · Grad-CAM · Vanilla Saliency · LIME · XAI Validation Matrices · Per-class Metrics · Ablation · Misclassification Analysis · Bootstrap CI · McNemar · ANOVA · AdamW + Cosine Decay**

---
**Proposed model:** Custom CNN trained from scratch (128×128 input, residual blocks, GAP head)  
**Baselines:** VGG16 · VGG19 · ResNet50 (ImageNet pretrained, 224×224 input, two-phase fine-tuning)  
**Dataset:** Masoud Nickparvar v2 — pre-balanced, 4 classes (Glioma · Meningioma · No Tumour · Pituitary)

In [ ]:
%pip install -q statsmodels tf-keras-vis lime scikit-image
%pip list | grep -E 'tensorflow|keras|lime|tf-keras|statsmodels|scikit'
print("\n============================================================")
print("  CELL 1  | Install dependencies")
print("============================================================")


# Configuration

- `RUN_MODE = 'debug'` → fast smoke-test (small data, 3 epochs, proposed model only, minimal XAI)
- `RUN_MODE = 'full'`  → full pipeline (all 4 models, 100 epochs, complete XAI & statistics)

**Key hyperparameters:**

| Parameter | Proposed CNN | Baselines |
|---|---|---|
| Input size | 128 × 128 | 224 × 224 |
| Batch size | 32 | 32 |
| Optimiser | AdamW + CosineDecay | Adam (lr=1e-3) |
| Loss | Categorical cross-entropy | Categorical cross-entropy |
| Max epochs | 100 | 100 + 33 (fine-tune) |
| Early stop patience | 15 | 8 |

> **Note:** Dataset is pre-balanced — no oversampling, no Focal Loss, no class weights needed.

In [ ]:
print("\n============================================================")
print("  CELL 3  | Imports & Configuration")
print("============================================================")
import os, logging, warnings, gc, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['PYTHONHASHSEED'] = '42'
logging.getLogger('tensorflow').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks, Input
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization, Activation,
    Flatten, Dense, Dropout
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras import Model, Model as KerasModel
from tensorflow.keras.applications import VGG16, VGG19, ResNet50

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    precision_recall_curve, auc, accuracy_score, f1_score,
    precision_score, recall_score, cohen_kappa_score
)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import label_binarize
import scipy.stats as stats
from scipy.stats import wilcoxon, f_oneway, spearmanr

from lime import lime_image
from skimage.segmentation import mark_boundaries

from tf_keras_vis.gradcam import Gradcam
from tf_keras_vis.saliency import Saliency
from tf_keras_vis.utils.scores import CategoricalScore

# ── Reproducibility
RANDOM_SEED = 42
def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
set_seed()


RUN_MODE = 'full'   # 'debug' | 'full'

IS_DEBUG = RUN_MODE == 'debug'

# Dataset (Masoud Nickparvar v2) is already balanced — single training regime
REGIMES = ['standard']

MAX_EPOCHS    = 3    if IS_DEBUG else 100
DEBUG_SAMPLES = 40   # 10 per class — stratified, fast
N_XAI         = 4   if IS_DEBUG else 12
N_FAITH       = 4   if IS_DEBUG else 12
N_SMOOTH      = 5   if IS_DEBUG else 30
N_INS_STEPS   = 10  if IS_DEBUG else 30
N_BOOTSTRAP   = 50  if IS_DEBUG else 1000
LIME_SAMPLES  = 50  if IS_DEBUG else 300

# ── Paths & hyperparams
DATASET_PATH    = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/'
IMAGE_SIZE         = (224, 224)   # baselines (ImageNet pretrained)
IMAGE_SIZE_PROPOSE = (128, 128)   # proposed scratch CNN — fewer params, less overfit
BATCH_SIZE      = 32        # increased for smoother gradients (scratch CNN)
LR_INIT         = 1e-3
LR_FINETUNE     = 1e-5
DROPOUT_RATE    = 0.3
L2_REG          = 1e-3
FINETUNE_LAYERS = 20

for d in ['results', 'plots', 'xai_outputs']:
    os.makedirs(d, exist_ok=True)

# ── Proposed model identity (used for comparison in stats/viz/export)
# Proposed model — propose CNN trained from scratch
# Baselines: VGG16, VGG19, ResNet50 (pretrained transfer learning)
PROPOSED_MODEL = 'propose_cnn'


print(f'RUN_MODE : {RUN_MODE}')
print(f'IS_DEBUG : {IS_DEBUG}')
print(f'REGIMES  : {REGIMES}')
print(f'MAX_EPOCHS: {MAX_EPOCHS}')


## Plot Style (journal quality)
Matplotlib rcParams configured for publication-ready figures (300 DPI TIFF output).

In [ ]:
print("\n============================================================")
print("  CELL 5  | Plot style")
print("============================================================")
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'figure.figsize': (6, 4),
    'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'savefig.bbox': 'tight', 'savefig.format': 'tiff',
})
sns.set_style('whitegrid')
print('Plot style set (300 DPI TIFF).')


## System Information
Reports GPU availability, TensorFlow version, and compute environment.

In [ ]:
print("\n============================================================")
print("  CELL 7  | System information")
print("============================================================")
import platform, psutil
for k, v in {
    'OS': platform.system(), 'Python': platform.python_version(),
    'TensorFlow': tf.__version__,
    'GPUs': [g.name for g in tf.config.list_physical_devices('GPU')],
    'RAM_GB': round(psutil.virtual_memory().total / 1e9, 1),
}.items(): print(f'{k}: {v}')


# Data Loading

Loads the **Masoud Nickparvar v2** brain tumour MRI dataset from disk.

- **Classes (4):** Glioma · Meningioma · No Tumour · Pituitary
- **Source split:** Official `Training/` and `Testing/` folders used as-is
- **Debug mode:** Stratified sub-sample of 10 images per class for fast iteration

> Dataset is already class-balanced — no oversampling step required.

In [ ]:
print("\n============================================================")
print("  CELL 9  | Data loading")
print("============================================================")
train_dir = os.path.join(DATASET_PATH, 'Training')
test_dir  = os.path.join(DATASET_PATH, 'Testing')

class_names = sorted(os.listdir(train_dir))
num_classes = len(class_names)
print('Classes:', class_names)

def load_paths_labels(directory):
    paths, labels = [], []
    for i, cls in enumerate(class_names):
        ps = sorted(glob.glob(os.path.join(directory, cls, '*.jpg')))
        paths.extend(ps); labels.extend([i] * len(ps))
    return np.array(paths), np.array(labels)

all_train_paths, all_train_labels = load_paths_labels(train_dir)
test_paths, test_labels           = load_paths_labels(test_dir)

if IS_DEBUG:
    keep = []
    per_cls = DEBUG_SAMPLES // num_classes
    for c in range(num_classes):
        idx = np.where(all_train_labels == c)[0]
        keep.extend(idx[:per_cls].tolist())
    all_train_paths  = all_train_paths[keep]
    all_train_labels = all_train_labels[keep]
    test_keep = []
    for c in range(num_classes):
        idx = np.where(test_labels == c)[0]
        test_keep.extend(idx[:5].tolist())
    test_paths  = test_paths[test_keep]
    test_labels = test_labels[test_keep]

print(f'Train pool: {len(all_train_paths)} | Test: {len(test_paths)}')
for i, cls in enumerate(class_names):
    print(f'  {cls}: train={np.sum(all_train_labels==i)}, test={np.sum(test_labels==i)}')


# Dataset Distribution

The Masoud Nickparvar v2 dataset is **pre-balanced** across all 4 classes.

- No `RandomOverSampler` or `SMOTE` is applied
- Standard `categorical_crossentropy` loss used throughout
- No class weights required (`cw = None`)

In [ ]:
print("\n============================================================")
print("  CELL 11 | Dataset class distribution")
print("============================================================")

unique, counts = np.unique(all_train_labels, return_counts=True)
print("Training set class distribution:")
for cls_idx, cnt in zip(unique, counts):
    print(f"  {class_names[cls_idx]:12s}: {cnt} samples")

loss_fn = 'categorical_crossentropy'
cw      = None   # no class weights needed

print(f"\nLoss function : {loss_fn}")
print("Class weights : None (dataset is balanced)")


# Stratified Train / Validation Split (80 / 10 / 10)

The official `Testing/` folder provides the fixed **10% test set** (never seen during training).  
The remaining `Training/` pool is split **90/10** into train and validation sets using stratified sampling.

| Split | Source | Purpose |
|---|---|---|
| Train (80%) | `Training/` × 0.90 | Model fitting |
| Val (10%) | `Training/` × 0.10 | Early stopping & checkpointing |
| Test (10%) | `Testing/` (fixed) | Final honest evaluation |

> Validation set is **never** used for final accuracy reporting — prevents data leakage.

In [ ]:
print("\n============================================================")
print("  CELL 13 | Dataset pool summary (pre-regime split)")
print("============================================================")

print(f"Raw training pool : {len(all_train_paths)} samples")
print(f"Test set (fixed)  : {len(test_paths)} samples")
print("\nClass distribution in raw training pool:")
for i, cls in enumerate(class_names):
    n_tr = int(np.sum(all_train_labels == i))
    n_te = int(np.sum(test_labels == i))
    print(f"  {cls}: train={n_tr}  test={n_te}")
print("\nNote: 90/10 val split and oversampling (balanced) are applied per-regime in Cell 23.")


# tf.data Pipeline — Preprocessing

**No augmentation applied** — clean preprocessing only on all splits.

Preprocessing chain (applied identically to train, val, and test):

1. Decode JPEG → grayscale (1 channel)
2. Resize to target resolution
3. **CLAHE** (`clipLimit=2.0, tileGridSize=8×8`) — local contrast enhancement
4. **Bilateral filter** (`d=2, σ=50`) — edge-preserving noise reduction
5. **COLORMAP_BONE** pseudo-colour mapping (grayscale → 3-channel)
6. Cast to float32 and normalise to **[0, 1]** (`/ 255.0`)

> `per_image_standardization` removed — avoids clipping bias when pixel values go negative after standardisation.

**Two separate pipelines:**
- **Proposed CNN:** resizes to `128 × 128` (reduces parameter count, less overfitting on ~5K samples)
- **Baselines:** resizes to `224 × 224` (required for ImageNet pretrained weights)

In [ ]:
print("\n============================================================")
print("  CELL 15 | tf.data pipeline (preprocess only — no augmentation)")
print("============================================================")

# ── Item 2: removed per_image_standardization — clean [0,1] range only
# ── Item 10: dual preprocessing functions (128x128 proposed / 224x224 baselines)

def preprocess_image(image_path, label, img_size=None):
    """CLAHE → bilateral filter → COLORMAP_BONE → /255 → [0,1]."""
    if img_size is None:
        img_size = IMAGE_SIZE
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=1)
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.image.resize(image, img_size)
    image = tf.squeeze(image, axis=-1)
    image = tf.numpy_function(
        lambda x: cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(
            (x*255).astype(np.uint8)), [image], tf.uint8)
    image = tf.numpy_function(
        lambda x: cv2.bilateralFilter(x, d=2, sigmaColor=50, sigmaSpace=50),
        [image], tf.uint8)
    image = tf.numpy_function(
        lambda x: cv2.applyColorMap(x, cv2.COLORMAP_BONE), [image], tf.uint8)
    # Item 2: /255 only — no per_image_standardization
    image = tf.cast(image, tf.float32) / 255.0
    image.set_shape(img_size + (3,))
    return image, label


def preprocess_image_propose(image_path, label):
    """128x128 pipeline for proposed scratch CNN."""
    return preprocess_image(image_path, label, img_size=IMAGE_SIZE_PROPOSE)


def preprocess_image_baseline(image_path, label):
    """224x224 pipeline for pretrained baselines."""
    return preprocess_image(image_path, label, img_size=IMAGE_SIZE)


def create_dataset(paths, labels, shuffle=True, augment_data=False,
                   is_propose=False):
    """Build tf.data pipeline. augment_data param kept for API compatibility
    but augmentation is fully disabled (user request).
    is_propose=True  → IMAGE_SIZE_PROPOSE (128x128)
    is_propose=False → IMAGE_SIZE (224x224)
    """
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=RANDOM_SEED)
    if is_propose:
        ds = ds.map(preprocess_image_propose, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(preprocess_image_baseline, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

print("Preprocessing pipelines defined (no augmentation):")
print(f"  Proposed  : {IMAGE_SIZE_PROPOSE} — CLAHE + BilateralFilter + COLORMAP_BONE + /255")
print(f"  Baselines : {IMAGE_SIZE} — same preprocessing chain")
print("  Item 2: per_image_standardization REMOVED")
print("  Item 10: dual image-size pipelines active")


# Loss Function

**`categorical_crossentropy`** used for all models.

- Dataset is pre-balanced → no Focal Loss required
- No class weights (`cw = None`)
- Consistent loss function enables fair comparison across all 4 models

In [ ]:
print("\n============================================================")
print("  CELL 17 | Loss function")
print("============================================================")
print("Loss: categorical_crossentropy (dataset is pre-balanced, no focal loss needed)")


# Model Definitions

## Proposed Model — Custom CNN (trained from scratch)

| Improvement | Detail |
|---|---|
| **Architecture** | 5 residual blocks (32→64→128→256→512 filters) |
| **Pooling** | GlobalAveragePooling2D (replaces Flatten — avoids 25K-dim head) |
| **Skip connections** | Identity + projection shortcuts on every block |
| **Spatial dropout** | SpatialDropout2D(0.1–0.2) on deep blocks (3–5) |
| **Head dropout** | Dropout(0.5) after each Dense layer |
| **Initialiser** | HeNormal (optimal for Swish activation) |
| **L2 regularisation** | Tapered: `1e-4` (early) → `5e-4` → `1e-3` (deep) |
| **Input size** | 128 × 128 × 3 |

## Baseline Models — Transfer Learning (ImageNet pretrained)

| Model | Pool | Trainable head | Input |
|---|---|---|---|
| VGG16 | Flatten + Dense(256) | Yes | 224 × 224 |
| VGG19 | Flatten + Dense(256) | Yes | 224 × 224 |
| ResNet50 | GlobalAveragePooling2D | Yes | 224 × 224 |

All baseline backbones are **frozen** initially; top layers are unfrozen in Phase 2 fine-tuning.

In [ ]:
print("\n============================================================")
print("  CELL 19 | Model definitions")
print("============================================================")
from tensorflow.keras.layers import (
    GlobalAveragePooling2D, SpatialDropout2D, Add
)
from tensorflow.keras.initializers import HeNormal

HE = HeNormal(seed=RANDOM_SEED)   # Item 7: HeNormal init for swish/ReLU activations


# ─────────────────────────────────────────────────────────────────
# Item 4: Residual block helper
# ─────────────────────────────────────────────────────────────────
def residual_block(x, filters, l2_reg=0.001, spatial_dropout_rate=0.0):
    """Two Conv2D + BN + Swish with identity shortcut.
    Projection shortcut applied when channel dims differ.
    Item 8: l2_reg passed per-block so early blocks use lower regularisation.
    Item 6: optional SpatialDropout2D inside block for deep layers.
    """
    shortcut = x
    # Branch
    x = Conv2D(filters, (3, 3), padding='same',
               kernel_regularizer=l2(l2_reg),
               kernel_initializer=HE)(x)          # Item 7: HeNormal
    x = BatchNormalization()(x)
    x = Activation('swish')(x)
    x = Conv2D(filters, (3, 3), padding='same',
               kernel_regularizer=l2(l2_reg),
               kernel_initializer=HE)(x)          # Item 7: HeNormal
    x = BatchNormalization()(x)
    # Item 6: spatial dropout inside deep blocks
    if spatial_dropout_rate > 0:
        x = SpatialDropout2D(spatial_dropout_rate)(x)
    # Projection shortcut when channel dims differ
    if shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, (1, 1), padding='same',
                          kernel_initializer=HE)(shortcut)
        shortcut = BatchNormalization()(shortcut)
    x = Add()([x, shortcut])
    x = Activation('swish')(x)
    return x


# ─────────────────────────────────────────────────────────────────
# Proposed model — custom CNN trained from scratch
# Items 1, 4, 6, 7, 8 all applied here
# ─────────────────────────────────────────────────────────────────
def create_propose_cnn(image_size=IMAGE_SIZE_PROPOSE, num_classes=4):
    """Proposed CNN with:
      - Item 1 : GlobalAveragePooling2D (not Flatten)
      - Item 4 : Residual connections on every conv block
      - Item 6 : SpatialDropout2D on deep blocks + Dropout(0.5) in head
      - Item 7 : HeNormal initialiser throughout
      - Item 8 : Tapered L2 — light (1e-4) early, heavy (1e-3) deep
    """
    if isinstance(image_size, (tuple, list)):
        input_shape = tuple(image_size) + (3,)
    else:
        input_shape = (image_size, image_size, 3)
    inputs = Input(shape=input_shape)

    # ── Stem: single conv to project to 32 channels
    x = Conv2D(32, (3, 3), padding='same',
               kernel_regularizer=l2(1e-4),        # Item 8: light L2 early
               kernel_initializer=HE)(inputs)       # Item 7: HeNormal
    x = BatchNormalization()(x)
    x = Activation('swish')(x)
    x = MaxPooling2D()(x)                           # 128→64

    # ── Block 1: 32 filters, light L2, no spatial dropout (early layer)
    x = residual_block(x, 32, l2_reg=1e-4, spatial_dropout_rate=0.0)   # Item 8
    x = MaxPooling2D()(x)                           # 64→32

    # ── Block 2: 64 filters, light L2
    x = residual_block(x, 64, l2_reg=1e-4, spatial_dropout_rate=0.0)   # Item 8
    x = MaxPooling2D()(x)                           # 32→16

    # ── Block 3: 128 filters, medium L2, begin spatial dropout
    x = residual_block(x, 128, l2_reg=5e-4, spatial_dropout_rate=0.1)  # Items 6,8
    x = MaxPooling2D()(x)                           # 16→8

    # ── Block 4: 256 filters, heavier L2, spatial dropout
    x = residual_block(x, 256, l2_reg=1e-3, spatial_dropout_rate=0.2)  # Items 6,8
    x = MaxPooling2D()(x)                           # 8→4

    # ── Block 5: 512 filters, heaviest L2, spatial dropout
    x = residual_block(x, 512, l2_reg=1e-3, spatial_dropout_rate=0.2)  # Items 6,8

    # ── Item 1: GlobalAveragePooling2D instead of Flatten
    # Reduces 4×4×512=8192 → 512, far fewer params, much less overfit
    x = GlobalAveragePooling2D()(x)

    # ── Classification head — Item 6: Dropout(0.5), Item 7: HeNormal
    x = Dense(256, activation='swish',
               kernel_regularizer=l2(1e-3),
               kernel_initializer=HE)(x)
    x = Dropout(0.5)(x)                             # Item 6: 0.3→0.5
    x = Dense(128, activation='swish',
               kernel_regularizer=l2(1e-3),
               kernel_initializer=HE)(x)
    x = Dropout(0.5)(x)                             # Item 6: 0.3→0.5
    outputs = Dense(num_classes, activation='softmax',
                    kernel_initializer=HE)(x)       # Item 7: HeNormal on output too

    return Model(inputs, outputs, name="propose_cnn")


# ─────────────────────────────────────────────────────────────────
# Baseline models — ImageNet pretrained transfer learning (unchanged)
# ─────────────────────────────────────────────────────────────────
def _transfer_model(base_cls, name, pool='gap', nc=None):
    nc = nc or num_classes
    base = base_cls(weights='imagenet', include_top=False,
                    input_shape=IMAGE_SIZE + (3,))
    base.trainable = False
    x = base.output
    if pool == 'flatten':
        x = layers.Flatten()(x)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.5)(x)
    else:
        x = layers.GlobalAveragePooling2D()(x)
    out = layers.Dense(nc, activation='softmax')(x)
    return KerasModel(inputs=base.input, outputs=out, name=name)

def create_vgg16(**kw):    return _transfer_model(VGG16,    'vgg16',    pool='flatten', **kw)
def create_vgg19(**kw):    return _transfer_model(VGG19,    'vgg19',    pool='flatten', **kw)
def create_resnet50(**kw): return _transfer_model(ResNet50, 'resnet50',                 **kw)

if IS_DEBUG:
    models_dict = {'propose_cnn': lambda: create_propose_cnn(IMAGE_SIZE_PROPOSE, num_classes)}
else:
    models_dict = {
        'propose_cnn': lambda: create_propose_cnn(IMAGE_SIZE_PROPOSE, num_classes),  # proposed
        'vgg16':       lambda: create_vgg16(nc=num_classes),                          # baseline
        'vgg19':       lambda: create_vgg19(nc=num_classes),                          # baseline
        'resnet50':    lambda: create_resnet50(nc=num_classes),                        # baseline
    }

print('Models:', list(models_dict.keys()))
print(f'Proposed model: {PROPOSED_MODEL}  (trained from scratch, 128x128 input)')
print('Baselines     : vgg16, vgg19, resnet50  (ImageNet pretrained, 224x224 input)')


# Training Engine

## Proposed CNN — Two-component strategy

| Component | Detail |
|---|---|
| **Optimiser** | `AdamW` (decoupled weight decay `1e-4`) |
| **LR schedule** | `CosineDecay`: `1e-3` → `1e-6` over `MAX_EPOCHS` steps |
| **Early stopping** | `patience=15`, monitors `val_loss` |
| **No ReduceLROnPlateau** | Incompatible with `LearningRateSchedule` — CosineDecay handles LR automatically |

## Baselines — Standard fine-tuning

| Phase | Detail |
|---|---|
| **Phase 1** | Frozen backbone, Adam (`lr=1e-3`), `patience=8` |
| **Phase 2** | Unfreeze top 20 layers, Adam (`lr=1e-5`), ~33 epochs |
| **ReduceLROnPlateau** | `factor=0.5, patience=4, min_lr=1e-7` |

`ModelCheckpoint` saves best `val_loss` checkpoint for all models.

In [ ]:
print("\n============================================================")
print("  CELL 21 | Training engine (compile / train / fine-tune / metrics)")
print("============================================================")

# ── Item 5: AdamW + Cosine Decay for proposed scratch CNN
def compile_propose_model(model, total_steps):
    """Compile proposed CNN with AdamW + Cosine Decay schedule.
    Item 5: cosine decay prevents sharp LR drops that destabilise scratch training.
    AdamW decoupled weight decay is more effective than L2-via-Adam.
    """
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=1e-3,
        decay_steps=total_steps,
        alpha=1e-6           # floor LR — never fully zero
    )
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=lr_schedule,
        weight_decay=1e-4    # decoupled weight decay
    )
    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


# ── Standard compile for baselines (Adam, fixed LR)
def compile_model(model, lr=LR_INIT, loss_fn="categorical_crossentropy"):
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss=loss_fn,
        metrics=["accuracy"]
    )
    return model


# ── Item 9: separate callback sets — proposed needs more patience (scratch is slower)
def get_callbacks(model_name, suffix="", is_propose=False):
    """
    is_propose=True  → patience=15, NO ReduceLROnPlateau (CosineDecay handles LR)
    is_propose=False → patience=8,  ReduceLROnPlateau included (Adam flat LR)

    CosineDecay uses a LearningRateSchedule object — Keras blocks manual LR
    assignment, so ReduceLROnPlateau must be excluded for the proposed model.
    """
    patience_es = 15 if is_propose else 8    # Item 9
    base_cbs = [
        callbacks.EarlyStopping(monitor="val_loss", patience=patience_es,
                                restore_best_weights=True, verbose=1),
        callbacks.ModelCheckpoint(
            filepath=f"results/{model_name}{suffix}_best.keras",
            monitor="val_loss", save_best_only=True, verbose=0),
        callbacks.TerminateOnNaN(),
        callbacks.CSVLogger(f"results/{model_name}{suffix}_log.csv"),
    ]
    if not is_propose:
        # ReduceLROnPlateau only for baselines — incompatible with LearningRateSchedule
        base_cbs.insert(1, callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=4, min_lr=1e-7, verbose=1))
    return base_cbs


def train_model(model, train_ds, val_ds, epochs=MAX_EPOCHS,
                class_weights=None, is_propose=False):
    """Phase 1 training. is_propose flag routes to correct callbacks."""
    history = model.fit(
        train_ds, validation_data=val_ds, epochs=epochs,
        class_weight=class_weights,
        callbacks=get_callbacks(model.name, "_phase1", is_propose=is_propose),
        verbose=1
    )
    return model, history


def fine_tune_model(model, train_ds, val_ds, epochs=None, class_weights=None,
                    loss_fn="categorical_crossentropy"):
    """Phase 2: unfreeze top FINETUNE_LAYERS for pretrained baselines.
    Skipped for propose_cnn (no pretrained backbone to unfreeze).
    """
    if model.name == "propose_cnn":
        print("  [fine-tune] skipped: propose_cnn has no pretrained backbone.")
        return model

    backbone = None
    for layer in model.layers:
        if isinstance(layer, KerasModel):
            backbone = layer
            break
    if backbone is None:
        print("  [fine-tune] backbone not found, skipping.")
        return model

    for layer in backbone.layers[:-FINETUNE_LAYERS]:
        layer.trainable = False
    for layer in backbone.layers[-FINETUNE_LAYERS:]:
        layer.trainable = True

    n_ft_epochs = epochs if epochs else max(10, MAX_EPOCHS // 3)
    compile_model(model, lr=LR_FINETUNE, loss_fn=loss_fn)
    print(f"  [fine-tune] top {FINETUNE_LAYERS} layers unfrozen, LR={LR_FINETUNE}")
    model.fit(
        train_ds, validation_data=val_ds, epochs=n_ft_epochs,
        class_weight=class_weights,
        callbacks=get_callbacks(model.name, "_phase2", is_propose=False),
        verbose=1
    )
    return model


def compute_metrics(y_true, y_pred_classes, y_pred_probs):
    m = {}
    m["accuracy"]           = accuracy_score(y_true, y_pred_classes)
    m["f1_weighted"]        = f1_score(y_true, y_pred_classes, average="weighted", zero_division=0)
    m["f1_macro"]           = f1_score(y_true, y_pred_classes, average="macro",    zero_division=0)
    m["precision_weighted"] = precision_score(y_true, y_pred_classes, average="weighted", zero_division=0)
    m["precision_macro"]    = precision_score(y_true, y_pred_classes, average="macro",    zero_division=0)
    m["recall_weighted"]    = recall_score(y_true, y_pred_classes,    average="weighted", zero_division=0)
    m["recall_macro"]       = recall_score(y_true, y_pred_classes,    average="macro",    zero_division=0)
    m["kappa"]              = cohen_kappa_score(y_true, y_pred_classes)
    m["confusion_matrix"]   = confusion_matrix(y_true, y_pred_classes)

    cm = m["confusion_matrix"]
    per_class = {}
    for i, cls in enumerate(class_names):
        tp = cm[i, i]; fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp; tn = cm.sum() - tp - fn - fp
        per_class[cls] = {
            "sensitivity": round(float(tp / (tp + fn + 1e-8)), 4),
            "specificity":  round(float(tn / (tn + fp + 1e-8)), 4),
        }
    m["per_class"] = per_class

    try:
        m["roc_auc"] = roc_auc_score(y_true, y_pred_probs, multi_class="ovr")
    except Exception:
        m["roc_auc"] = None
    try:
        prs = [auc(*precision_recall_curve(
                   (y_true==i).astype(int), y_pred_probs[:,i])[1::-1])
               for i in range(y_pred_probs.shape[1])]
        m["pr_auc"] = float(np.mean(prs))
    except Exception:
        m["pr_auc"] = None
    return m

print("Training engine defined:")
print("  compile_propose_model(model, total_steps)  — AdamW + CosineDecay (Item 5)")
print("  compile_model(model, lr, loss_fn)          — Adam for baselines")
print("  train_model(..., is_propose)               — patience 15/8 (Item 9)")
print("  fine_tune_model(...)                       — Phase 2 for baselines")
print("  compute_metrics(y_true, y_pred_classes, y_pred_probs)")


# Train All Models

Training loop iterates over all 4 models in `models_dict`:

1. **`propose_cnn`** — AdamW + CosineDecay, 128×128 pipeline, patience=15
2. **`vgg16`** — Adam Phase 1 → fine-tune Phase 2, 224×224 pipeline
3. **`vgg19`** — Adam Phase 1 → fine-tune Phase 2, 224×224 pipeline
4. **`resnet50`** — Adam Phase 1 → fine-tune Phase 2, 224×224 pipeline

Each model:
- Trained on `train_ds` → validated on `val_ds`
- Final metrics computed on held-out `test_ds` (never seen during training)
- Saved to `results/<model_name>/model.keras`

In [ ]:
print("\n============================================================")
print("  CELL 23 | Train all models")
print("============================================================")

all_trained_models = {}
all_results        = {}
all_test_preds     = {}
all_test_probs     = {}

y_test_oh = tf.keras.utils.to_categorical(test_labels, num_classes)

# Single stratified 90/10 train/val split
tr_p, val_p, tr_l, val_l = train_test_split(
    all_train_paths, all_train_labels,
    test_size=0.111, stratify=all_train_labels,
    random_state=RANDOM_SEED
)
y_tr_oh  = tf.keras.utils.to_categorical(tr_l,  num_classes)
y_val_oh = tf.keras.utils.to_categorical(val_l, num_classes)

# ── Item 10: separate tf.data pipelines — 128x128 for proposed, 224x224 for baselines
# ── No augmentation (user request) — augment_data=False throughout
train_ds_propose  = create_dataset(tr_p,       y_tr_oh,  shuffle=True,  augment_data=False, is_propose=True)
val_ds_propose    = create_dataset(val_p,       y_val_oh, shuffle=False, augment_data=False, is_propose=True)
test_ds_propose   = create_dataset(test_paths, y_test_oh, shuffle=False, augment_data=False, is_propose=True)

train_ds_baseline = create_dataset(tr_p,       y_tr_oh,  shuffle=True,  augment_data=False, is_propose=False)
val_ds_baseline   = create_dataset(val_p,       y_val_oh, shuffle=False, augment_data=False, is_propose=False)
test_ds_baseline  = create_dataset(test_paths, y_test_oh, shuffle=False, augment_data=False, is_propose=False)

# Expose for ablation & XAI cells
train_paths = tr_p;  train_labels = tr_l
val_paths   = val_p; val_labels   = val_l
y_train_oh  = y_tr_oh
train_ds    = train_ds_baseline   # alias for XAI / ablation cells
val_ds      = val_ds_baseline
test_ds     = test_ds_baseline

print(f"Train={len(tr_p)}  Val={len(val_p)}  Test={len(test_paths)}")
print(f"Proposed pipeline : {IMAGE_SIZE_PROPOSE}  Baseline pipeline: {IMAGE_SIZE}")
print("Augmentation: disabled")

# ── Item 5: total_steps for cosine decay (proposed only)
steps_per_epoch_propose = max(1, len(tr_p) // BATCH_SIZE)
total_steps_propose     = steps_per_epoch_propose * MAX_EPOCHS

for model_name, model_fn in models_dict.items():
    is_propose = (model_name == PROPOSED_MODEL)

    # Select correct dataset pair
    _train_ds = train_ds_propose if is_propose else train_ds_baseline
    _val_ds   = val_ds_propose   if is_propose else val_ds_baseline
    _test_ds  = test_ds_propose  if is_propose else test_ds_baseline

    print(f"\n  [{model_name}] Phase 1 — training")
    model = model_fn()

    # Item 5: AdamW + CosineDecay for proposed; plain Adam for baselines
    if is_propose:
        compile_propose_model(model, total_steps_propose)
    else:
        compile_model(model, lr=LR_INIT, loss_fn=loss_fn)

    model, _ = train_model(model, _train_ds, _val_ds,
                           epochs=MAX_EPOCHS, class_weights=cw,
                           is_propose=is_propose)

    # Phase 2 fine-tune only for pretrained baselines
    if not IS_DEBUG and not is_propose:
        print(f"  [{model_name}] Phase 2 — fine-tune top {FINETUNE_LAYERS} layers")
        model = fine_tune_model(model, _train_ds, _val_ds,
                                class_weights=cw, loss_fn=loss_fn)

    y_prob         = model.predict(_test_ds, verbose=0)
    y_pred_classes = np.argmax(y_prob, axis=1)
    metrics        = compute_metrics(test_labels, y_pred_classes, y_prob)

    all_trained_models[model_name] = model
    all_results[model_name]        = metrics
    all_test_preds[model_name]     = y_pred_classes
    all_test_probs[model_name]     = y_prob

    model_dir = os.path.join("results", model_name)
    os.makedirs(model_dir, exist_ok=True)
    model.save(os.path.join(model_dir, "model.keras"))

    print(f"  [{model_name}] "
          f"Acc={metrics['accuracy']*100:.2f}%  "
          f"F1w={metrics['f1_weighted']*100:.2f}%  "
          f"F1m={metrics['f1_macro']*100:.2f}%  "
          f"Kappa={metrics['kappa']:.4f}")

# Convenience aliases
all_metrics    = dict(all_results)
trained_models = dict(all_trained_models)
model_preds    = dict(all_test_preds)
model_probs    = dict(all_test_probs)

# Summary table
rows = []
for mn, m in all_metrics.items():
    rows.append({
        "model":                     mn,
        "is_proposed":               mn == PROPOSED_MODEL,
        "accuracy (in %)":           round(m["accuracy"]*100, 2),
        "precision_weighted (in %)": round(m["precision_weighted"]*100, 2),
        "recall_weighted (in %)":    round(m["recall_weighted"]*100, 2),
        "f1_weighted (in %)":        round(m["f1_weighted"]*100, 2),
        "f1_macro (in %)":           round(m["f1_macro"]*100, 2),
        "kappa":                     round(m["kappa"], 4),
        "roc_auc":                   round(m["roc_auc"], 4) if m.get("roc_auc") else None,
        "pr_auc":                    round(m["pr_auc"],  4) if m.get("pr_auc")  else None,
    })
summary_df = pd.DataFrame(rows)
print("\n=== Summary — All Models ===")
print(summary_df.to_string(index=False))
summary_df.to_csv("results/summary.csv", index=False)

pc_rows = []
for mn, m in all_metrics.items():
    for cls, vals in m["per_class"].items():
        pc_rows.append({"model": mn, "class": cls, **vals})
pd.DataFrame(pc_rows).to_csv("results/per_class_sensitivity_specificity.csv", index=False)

best_key_str     = max(all_metrics, key=lambda k: all_metrics[k]["accuracy"])
best_model_name  = best_key_str
best_model       = trained_models[best_key_str]
best_model_preds = model_preds[best_key_str]
best_model_probs = model_probs[best_key_str]
print(f"\nBest model: {best_key_str}  Acc={all_metrics[best_key_str]['accuracy']*100:.2f}%")

split_df = pd.DataFrame({
    "Class": class_names,
    "Train": [int(np.sum(tr_l == i)) for i in range(num_classes)],
    "Val":   [int(np.sum(val_l == i)) for i in range(num_classes)],
    "Test":  [int(np.sum(test_labels == i)) for i in range(num_classes)],
})
split_df.to_csv("results/dataset_split.csv", index=False)
gc.collect()

print(f'\n{"="*60}')
print(f'  PROPOSED MODEL: {PROPOSED_MODEL}')
print(f'{"="*60}')
if PROPOSED_MODEL in all_results:
    pm = all_results[PROPOSED_MODEL]
    print(f'  Proposed ({PROPOSED_MODEL}) — Acc={pm["accuracy"]*100:.2f}%  '
          f'F1m={pm["f1_macro"]*100:.2f}%  Kappa={pm["kappa"]:.4f}')
    for mn, m in sorted(all_results.items()):
        if mn == PROPOSED_MODEL:
            continue
        delta_acc = (pm["accuracy"] - m["accuracy"]) * 100
        delta_f1  = (pm["f1_macro"] - m["f1_macro"]) * 100
        print(f'  vs {mn:12s}: acc {delta_acc:+.2f}%  f1m {delta_f1:+.2f}%')
else:
    print(f'  [warning] {PROPOSED_MODEL} not found in results.')


# Ablation Study

Systematic component contribution analysis on the **proposed CNN only**.

Each ablation removes one design choice and retrains from scratch to measure the accuracy drop:

| Configuration | Component removed |
|---|---|
| Full model (all components) | — baseline |
| No CLAHE preprocessing | CLAHE enhancement |
| No augmentation | (N/A — augmentation disabled) |
| No fine-tuning (frozen backbone only) | Phase 2 fine-tune (N/A for scratch CNN) |

> Ablation models use `IMAGE_SIZE_PROPOSE` (128×128) to match the proposed model's pipeline.

In [ ]:
print("\n============================================================")
print("  CELL 25 | Ablation study")
print("============================================================")


ablation_rows = []

ablation_best_key  = PROPOSED_MODEL   # ablation always on proposed model
ablation_model_name = PROPOSED_MODEL
ablation_regime     = 'standard'
print(f'Ablation reference: proposed model = {ablation_best_key}')

# ── Full model baseline
best_acc = all_metrics[ablation_best_key]['accuracy']
ablation_rows.append({
    'Configuration': 'Full model (all components)',
    'Regime': ablation_regime,
    'Model': ablation_model_name,
    'Accuracy (in %)': round(best_acc * 100, 2),
    'Delta vs Full (%)': 0.0,
})

# ── Ablation 1: No CLAHE (plain resize only)
def preprocess_no_clahe(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.image.per_image_standardization(image)
    image.set_shape(IMAGE_SIZE + (3,))
    return image, label

def create_plain_ds(paths, labels):
    """Ablation: no CLAHE — uses IMAGE_SIZE_PROPOSE (128x128) for propose CNN."""
    def _plain(image_path, label):
        image = tf.io.read_file(image_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.convert_image_dtype(image, tf.float32)
        image = tf.image.resize(image, IMAGE_SIZE_PROPOSE)
        image = tf.image.per_image_standardization(image)
        image.set_shape(IMAGE_SIZE_PROPOSE + (3,))
        return image, label
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(_plain, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print('Ablation 1: No CLAHE preprocessing...')
abl1_train_ds = create_plain_ds(train_paths, y_train_oh)
abl1_val_ds   = create_plain_ds(val_paths,   y_val_oh)
abl1_test_ds  = create_plain_ds(test_paths,  y_test_oh)

abl1_m = models_dict[ablation_model_name]()
compile_model(abl1_m, lr=LR_INIT, loss_fn=loss_fn)
abl1_m, _ = train_model(
    abl1_m, abl1_train_ds, abl1_val_ds,
    epochs=MAX_EPOCHS,
    class_weights=cw
    )
abl1_pred  = abl1_m.predict(abl1_test_ds, verbose=0)
abl1_acc   = accuracy_score(test_labels, np.argmax(abl1_pred, 1))
ablation_rows.append({
    'Configuration': 'No CLAHE preprocessing',
    'Regime': ablation_regime,
    'Model': ablation_model_name,
    'Accuracy (in %)': round(abl1_acc * 100, 2),
    'Delta vs Full (%)': round((abl1_acc - best_acc) * 100, 2),
})
print(f'  No-CLAHE Acc: {abl1_acc*100:.2f}%')

# ── Ablation 2: No augmentation
print('Ablation 2: No augmentation...')
abl2_train_ds = create_dataset(train_paths, y_train_oh, shuffle=True, augment_data=False, is_propose=True)
abl2_m = models_dict[ablation_model_name]()
compile_model(abl2_m, lr=LR_INIT, loss_fn=loss_fn)
abl2_m, _ = train_model(
    abl2_m, abl2_train_ds, val_ds_propose,
    epochs=MAX_EPOCHS,
    class_weights=cw
    )
abl2_pred  = abl2_m.predict(test_ds_propose, verbose=0)
abl2_acc   = accuracy_score(test_labels, np.argmax(abl2_pred, 1))
ablation_rows.append({
    'Configuration': 'No augmentation',
    'Regime': ablation_regime,
    'Model': ablation_model_name,
    'Accuracy (in %)': round(abl2_acc * 100, 2),
    'Delta vs Full (%)': round((abl2_acc - best_acc) * 100, 2),
})
print(f'  No-Aug Acc: {abl2_acc*100:.2f}%')

# ── Ablation 3: No fine-tuning (phase 1 only)
print('Ablation 3: No fine-tuning (frozen backbone only)...')
abl3_m = models_dict[ablation_model_name]()
compile_model(abl3_m, lr=LR_INIT, loss_fn=loss_fn)
abl3_m, _ = train_model(
    abl3_m, train_ds_propose, val_ds_propose,
    epochs=MAX_EPOCHS,
    class_weights=cw
    )
# do NOT call fine_tune_model
abl3_pred = abl3_m.predict(test_ds_propose, verbose=0)
abl3_acc  = accuracy_score(test_labels, np.argmax(abl3_pred, 1))
ablation_rows.append({
    'Configuration': 'No fine-tuning (frozen backbone only)',
    'Regime': ablation_regime,
    'Model': ablation_model_name,
    'Accuracy (in %)': round(abl3_acc * 100, 2),
    'Delta vs Full (%)': round((abl3_acc - best_acc) * 100, 2),
})
print(f'  No-FT Acc: {abl3_acc*100:.2f}%')

ablation_df = pd.DataFrame(ablation_rows)
print('\n=== Ablation Study ===')
print(ablation_df.to_string(index=False))
ablation_df.to_csv('results/ablation_study.csv', index=False)


# XAI Methods

Three complementary attribution methods are applied to all models:

| Method | Type | What it shows |
|---|---|---|
| **Grad-CAM** | Gradient × activation | Class-discriminative spatial regions |
| **Vanilla Saliency** | Input gradient | Pixel-level sensitivity to class score |
| **LIME** | Perturbation-based | Superpixel importance via local linear model |

> SHAP removed (computationally prohibitive).  
> ScoreCAM and SmoothGrad removed (replaced by the above three-method panel for XAI consistency).

In [ ]:
print("\n============================================================")
print("  CELL 27 | XAI helpers (Grad-CAM / Saliency / LIME)")
print("============================================================")

def to_functional(model):
    if not isinstance(model, KerasModel):
        raise TypeError(f'Expected a Keras model, got: {type(model)}')
    return model

def find_penultimate_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, KerasModel):
            nested = find_penultimate_conv_layer(layer)
            if nested is not None:
                return nested
        if isinstance(layer, (
            layers.Conv1D, layers.Conv2D, layers.Conv3D,
            layers.DepthwiseConv2D, layers.SeparableConv2D
        )):
            return layer.name
        try:
            output_shape = getattr(layer, 'output_shape', None)
            if output_shape is not None and len(output_shape) == 4:
                return layer.name
        except Exception:
            pass
    return None

def _to_nhw(maps):
    arr = np.asarray(maps)
    if arr.ndim == 2: return arr[None, ...]
    if arr.ndim == 4: return np.max(arr, axis=-1)
    if arr.ndim != 3: raise ValueError(f'Unsupported attribution shape: {arr.shape}')
    return arr

def _norm_maps(cam):
    cam = _to_nhw(cam).astype(np.float32)
    lo = cam.min(axis=(1,2), keepdims=True)
    hi = cam.max(axis=(1,2), keepdims=True)
    return (cam - lo) / (hi - lo + 1e-8)

def _score_for(class_idx):
    if isinstance(class_idx, (list, tuple, np.ndarray)):
        return CategoricalScore(list(class_idx))
    return CategoricalScore([int(class_idx)])

def compute_gradcam(mf, images_np, class_idx):
    mf = to_functional(mf)
    penultimate_layer = find_penultimate_conv_layer(mf)
    call_kwargs = {}
    if penultimate_layer is not None:
        call_kwargs['penultimate_layer'] = penultimate_layer
    cam = Gradcam(mf, clone=False)(_score_for(class_idx), images_np, **call_kwargs)
    return _norm_maps(cam)

def compute_vanilla_saliency(mf, images_np, class_idx):
    mf = to_functional(mf)
    sal = Saliency(mf, clone=False)(_score_for(class_idx), images_np)
    return _norm_maps(sal)

def compute_lime(model, image_np, num_samples=LIME_SAMPLES, num_features=5):
    explainer = lime_image.LimeImageExplainer(random_state=RANDOM_SEED)
    exp = explainer.explain_instance(
        image_np.astype(np.float64),
        lambda x: model.predict(x, verbose=0),
        top_labels=1, hide_color=0, num_samples=num_samples
    )
    _, mask = exp.get_image_and_mask(
        exp.top_labels[0], positive_only=True,
        num_features=num_features, hide_rest=False
    )
    return mask.astype(np.float32)

def _display_ready_image(image_np):
    img = np.asarray(image_np, dtype=np.float32)
    lo, hi = float(np.min(img)), float(np.max(img))
    if hi - lo < 1e-8:
        return np.zeros_like(img, dtype=np.float32)
    return np.clip((img - lo) / (hi - lo), 0.0, 1.0)

def plot_xai_panel(image_np, gc, sal, lime_mask, title, save_path):
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    original = _display_ready_image(image_np)
    for ax, (img, ttl, cmap) in zip(axes, [
        (original,  'Original', 'gray'),
        (gc,        'Grad-CAM', 'jet'),
        (sal,       'Saliency', 'hot'),
        (lime_mask, 'LIME',     'bwr'),
    ]):
        ax.imshow(img, cmap=cmap)
        ax.set_title(ttl, fontsize=8)
        ax.axis('off')
    fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

## Run XAI on Test Samples

Generates Grad-CAM, Saliency, and LIME maps for up to `N_XAI` test images per class.

- Each model uses its own resolution-matched preprocessing (`128×128` for proposed, `224×224` for baselines)
- Panels saved as TIFF to `xai_outputs/<model_name>/xai_panel_<i>.tiff`
- Zero maps used as safe fallback if a method fails for a given model

In [ ]:
print("\n============================================================")
print("  CELL 29 | Run XAI on test samples")
print("============================================================")

per_cls_xai = max(1, N_XAI // num_classes)
xai_indices, xai_labels_list = [], []
for c in range(num_classes):
    idx = np.where(test_labels == c)[0][:per_cls_xai]
    xai_indices.extend(idx.tolist())
    xai_labels_list.extend([int(test_labels[j]) for j in idx])
print(f'XAI sample count: {len(xai_indices)} (from {num_classes} classes)')

xai_paths = test_paths[xai_indices]

def load_xai_images(paths, img_size):
    """Load and preprocess images to numpy for XAI (mirrors tf.data pipeline)."""
    imgs = []
    for p in paths:
        img = cv2.imread(p.decode() if isinstance(p, bytes) else p, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, img_size)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img = clahe.apply(img)
        img = cv2.bilateralFilter(img, d=2, sigmaColor=50, sigmaSpace=50)
        img = cv2.applyColorMap(img, cv2.COLORMAP_BONE)
        img = img.astype(np.float32) / 255.0
        imgs.append(img)
    return np.array(imgs, dtype=np.float32)

xai_store = {}
effective_lime_samples = min(LIME_SAMPLES, 120 if not IS_DEBUG else 50)

for model_name, model in trained_models.items():
    print(f'\nXAI: {model_name}')
    mf = to_functional(model)

    img_size = IMAGE_SIZE_PROPOSE if model_name == PROPOSED_MODEL else IMAGE_SIZE
    xai_images_np = load_xai_images(xai_paths, img_size)

    n, h, w = xai_images_np.shape[0], xai_images_np.shape[1], xai_images_np.shape[2]
    fallback = np.zeros((n, h, w), dtype=np.float32)
    gradcam_maps = saliency_maps = fallback.copy()

    try:
        gradcam_maps  = compute_gradcam(mf,          xai_images_np, xai_labels_list)
        saliency_maps = compute_vanilla_saliency(mf, xai_images_np, xai_labels_list)
    except Exception as e:
        print(f'  [warning] Grad-CAM/Saliency failed for {model_name}: {e}')
        print('  [warning] Using zero maps as fallback for this model.')

    try:
        lime_masks = [compute_lime(model, img, num_samples=effective_lime_samples) for img in xai_images_np]
        lime_masks = np.array(lime_masks, dtype=np.float32)
    except Exception as e:
        print(f'  [warning] LIME failed for {model_name}: {e}')
        print('  [warning] Using zero masks as fallback for this model.')
        lime_masks = fallback.copy()

    xai_store[model_name] = {
        'images':   xai_images_np,
        'gradcam':  gradcam_maps,
        'saliency': saliency_maps,
        'lime':     lime_masks,
    }

    xai_dir = os.path.join('xai_outputs', model_name)
    os.makedirs(xai_dir, exist_ok=True)
    for i in range(len(xai_images_np)):
        plot_xai_panel(
            xai_images_np[i],
            gradcam_maps[i], saliency_maps[i], lime_masks[i],
            title=f'{model_name} | idx={i} | True={class_names[xai_labels_list[i]]}',
            save_path=os.path.join(xai_dir, f'xai_panel_{i}.tiff')
        )

    gc.collect()

print('\nXAI complete.')


## Misclassification Error Analysis

XAI maps generated for the **top 8 misclassified** test samples of the best-performing model.

Reveals *where* the model attends when it predicts the wrong class — identifies:
- Spurious background features
- Class confusion patterns (e.g. meningioma ↔ glioma)
- Low-confidence ambiguous cases

Panels saved to `xai_outputs/misclassified/`.

In [ ]:
print("\n============================================================")
print("  CELL 31 | Misclassification error analysis")
print("============================================================")
# Use appropriate test_ds depending on which model is best
_best_test_ds = test_ds_propose if best_model_name == PROPOSED_MODEL else test_ds
all_probs  = best_model.predict(_best_test_ds, verbose=0)
all_preds  = np.argmax(all_probs, axis=1)
misclf_idx = np.where(all_preds != test_labels)[0]
print(f'Misclassified: {len(misclf_idx)} / {len(test_labels)} = {len(misclf_idx)/len(test_labels)*100:.1f}%')

N_MISC = min(8, len(misclf_idx))
misc_records = []
mf_best = to_functional(best_model)
misc_dir = os.path.join('xai_outputs', 'misclassified')
os.makedirs(misc_dir, exist_ok=True)

for rank, idx in enumerate(misclf_idx[:N_MISC]):
    _pp_fn = preprocess_image_propose if best_model_name == PROPOSED_MODEL else preprocess_image_baseline
    img_t, _ = _pp_fn(test_paths[idx], test_labels[idx])
    img_np   = img_t.numpy()
    true_lbl = int(test_labels[idx])
    pred_lbl = int(all_preds[idx])
    conf     = float(all_probs[idx][pred_lbl])

    img_batch = np.expand_dims(img_np, 0)
    h, w = img_np.shape[0], img_np.shape[1]
    zero_map = np.zeros((h, w), dtype=np.float32)

    try:
        gc_m  = compute_gradcam(mf_best,          img_batch, [true_lbl])[0]
        sal_m = compute_vanilla_saliency(mf_best,  img_batch, [true_lbl])[0]
    except Exception as e:
        print(f'  [warning] Misclassification XAI failed for idx={idx}: {e}')
        gc_m = sal_m = zero_map

    try:
        lime_m = compute_lime(best_model, img_np)
    except Exception as e:
        print(f'  [warning] Misclassification LIME failed for idx={idx}: {e}')
        lime_m = zero_map

    save_p = os.path.join(misc_dir, f'misc_{rank}_true{class_names[true_lbl]}_pred{class_names[pred_lbl]}.tiff')
    plot_xai_panel(img_np, gc_m, sal_m, lime_m,
        title=f'MISCLASSIFIED | True: {class_names[true_lbl]} -> Pred: {class_names[pred_lbl]} (conf={conf:.2f})',
        save_path=save_p)

    misc_records.append({
        'test_idx': int(idx), 'image_path': test_paths[idx],
        'true_class': class_names[true_lbl], 'pred_class': class_names[pred_lbl],
        'confidence': round(conf, 4), 'saved_panel': save_p
    })

pd.DataFrame(misc_records).to_csv('xai_outputs/misclassification_analysis.csv', index=False)
print(f'Misclassification XAI panels saved ({N_MISC} images).')

# XAI Validation Matrices

Quantitative faithfulness metrics computed for all models using Grad-CAM heatmaps:

| Metric | What it measures | Desired direction |
|---|---|---|
| **Comprehensiveness** | Confidence drop when top-25% important region masked | Higher = more faithful |
| **Sufficiency** | Confidence when only top-25% important region retained | Lower = more focused |
| **Insertion AUC** | Confidence rise as pixels revealed by importance rank | Higher = better |
| **Deletion AUC** | Confidence drop as pixels removed by importance rank | Lower = better |
| **Spearman ρ (Grad-CAM vs Saliency)** | Multi-method attribution agreement | Higher = more consistent |

Wilcoxon signed-rank test used to compare proposed vs each baseline on Comprehensiveness, Insertion AUC, and Deletion AUC.

In [ ]:
print("\n============================================================")
print("  CELL 33 | XAI validation matrices — all regimes & models")
print("============================================================")

def _topk_mask3d(heatmap, frac=0.25):
    flat = heatmap.flatten()
    k    = max(1, int(len(flat) * frac))
    thr  = np.sort(flat)[-k]
    mask = (heatmap >= thr).astype(np.float32)
    return np.stack([mask] * 3, axis=-1)

def comprehensiveness(model, img, lbl, hmap, baseline=0.0):
    m3 = _topk_mask3d(hmap)
    masked = img.copy(); masked[m3 == 1] = baseline
    pf = model.predict(img[None],    verbose=0)[0][lbl]
    pm = model.predict(masked[None], verbose=0)[0][lbl]
    return float(pf - pm)

def sufficiency(model, img, lbl, hmap, baseline=0.0):
    m3   = _topk_mask3d(hmap)
    kept = np.full_like(img, baseline); kept[m3 == 1] = img[m3 == 1]
    pf = model.predict(img[None],  verbose=0)[0][lbl]
    pk = model.predict(kept[None], verbose=0)[0][lbl]
    return float(pk / (pf + 1e-8))

def insertion_auc(model, img, lbl, hmap, n_steps=N_INS_STEPS, baseline=0.0):
    order = np.argsort(hmap.flatten())[::-1]
    H, W  = hmap.shape
    step  = max(1, H * W // n_steps)
    rev   = np.zeros(H * W, dtype=bool)
    confs = []
    for s in range(n_steps):
        rev[order[s * step:min((s + 1) * step, H * W)]] = True
        p  = np.ones_like(img) * baseline
        m3 = np.stack([rev.reshape(H, W)] * 3, axis=-1)
        p[m3] = img[m3]
        confs.append(float(model.predict(p[None], verbose=0)[0][lbl]))
    return float(np.trapz(confs) / n_steps)

def deletion_auc(model, img, lbl, hmap, n_steps=N_INS_STEPS, baseline=0.0):
    order = np.argsort(hmap.flatten())[::-1]
    H, W  = hmap.shape
    step  = max(1, H * W // n_steps)
    rem   = np.zeros(H * W, dtype=bool)
    confs = []
    for s in range(n_steps):
        rem[order[s * step:min((s + 1) * step, H * W)]] = True
        p  = img.copy()
        m3 = np.stack([rem.reshape(H, W)] * 3, axis=-1)
        p[m3] = baseline
        confs.append(float(model.predict(p[None], verbose=0)[0][lbl]))
    return float(np.trapz(confs) / n_steps)

def spearman_agreement(a, b):
    rho, pval = spearmanr(a.flatten(), b.flatten())
    return float(rho), float(pval)

print('Computing XAI validation matrices for ALL regime+model combinations...')
xai_val_records = []

for key_str, maps in xai_store.items():
    mn     = key_str   # keys are plain model names
    model  = all_trained_models[mn]
    gc_m   = maps['gradcam']
    sal_m  = maps['saliency']

    model_images = maps['images']
    for i in range(min(N_FAITH, len(model_images))):
        img = model_images[i]
        lbl = xai_labels_list[i]
        gc  = gc_m[i]
        sal = sal_m[i]

        comp             = comprehensiveness(model, img, lbl, gc)
        suff             = sufficiency(model, img, lbl, gc)
        ins              = insertion_auc(model, img, lbl, gc)
        dele             = deletion_auc(model, img, lbl, gc)
        rho_gc_sal, p_gc_sal = spearman_agreement(gc, sal)

        xai_val_records.append({
            'model':                    mn,
            'sample_idx':               i,
            'true_class':               class_names[lbl],
            'comprehensiveness':        comp,
            'sufficiency':              suff,
            'insertion_auc':            ins,
            'deletion_auc':             dele,
            'spearman_rho_gc_saliency': rho_gc_sal,
            'spearman_pval_gc_saliency': p_gc_sal,
        })

xai_val_df = pd.DataFrame(xai_val_records)
xai_val_df.to_csv('xai_outputs/xai_validation_metrics.csv', index=False)

pivot = xai_val_df.groupby(['model'])[[
    'comprehensiveness', 'sufficiency', 'insertion_auc', 'deletion_auc',
    'spearman_rho_gc_saliency']].mean().round(4)
print(pivot.to_string())

print(f'\n=== Proposed model XAI faithfulness: {PROPOSED_MODEL} vs baselines ===')
prop_rows = xai_val_df[
    (xai_val_df['model'] == PROPOSED_MODEL)
]
for key_str in xai_store:
    if key_str == PROPOSED_MODEL:
        continue
    base_rows = xai_val_df[
        xai_val_df['model'] == key_str
    ]
    for metric in ['comprehensiveness', 'insertion_auc', 'deletion_auc']:
        pv = prop_rows[metric].values
        bv = base_rows[metric].values
        n  = min(len(pv), len(bv))
        if n >= 2:
            stat, p = wilcoxon(pv[:n], bv[:n])
            direction = 'better' if pv[:n].mean() > bv[:n].mean() else 'worse'
            sig = '*SIG*' if p < 0.05 else ''
            print(f'  {metric:20s} vs {key_str}: p={p:.4f} {sig} proposed is {direction}')


# Statistical Validation

Rigorous statistical tests to support model comparison claims:

| Test | Purpose | Applied to |
|---|---|---|
| **Bootstrap CI (n=1000)** | 95% confidence interval on accuracy | All models |
| **McNemar's test** | Pairwise comparison on same test images | All model pairs |
| **Cohen's Kappa** | Agreement beyond chance (≥0.80 = strong) | All models |
| **ANOVA + Tukey HSD** | Do models differ in XAI comprehensiveness? | All models |
| **Wilcoxon signed-rank** | Insertion/Deletion AUC vs random baseline (0.5) | All models |
| **ECE** | Expected Calibration Error — confidence reliability | All models |
| **Robustness** | Accuracy under Gaussian noise (std=0.10) | All models |
| **McNemar (proposed vs each baseline)** | Proposed CNN significance over each baseline | Proposed vs 3 baselines |

In [ ]:
print("\n============================================================")
print("  CELL 35 | Statistical validation — all models")
print("============================================================")
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import ttest_rel

y_true = test_labels

# ── 1. Bootstrap CI (n=1000) — 95% confidence interval on accuracy
print('1. Bootstrap CI (n=1000)...')
boot_records = []
for mn, preds in model_preds.items():
    accs = [accuracy_score(
        y_true[idx := np.random.choice(len(y_true), len(y_true), replace=True)],
        preds[idx]) for _ in range(N_BOOTSTRAP)]
    ci_lo, ci_hi = np.percentile(accs, [2.5, 97.5])
    point = accuracy_score(y_true, preds)
    boot_records.append({
        'model':            mn,
        'is_proposed':      mn == PROPOSED_MODEL,
        'accuracy (in %)':  round(point * 100, 2),
        'ci_low_95 (in %)': round(ci_lo  * 100, 2),
        'ci_high_95 (in %)':round(ci_hi  * 100, 2),
    })
    print(f'  {mn}: {point*100:.2f}% [95% CI {ci_lo*100:.2f}–{ci_hi*100:.2f}]')
boot_df = pd.DataFrame(boot_records)
boot_df.to_csv('results/bootstrap_ci.csv', index=False)

# ── 2. McNemar — all pairwise comparisons
print('\n2. McNemar pairwise tests...')
mc_records = []
keys = list(model_preds.keys())
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        k1, k2 = keys[i], keys[j]
        c1 = (model_preds[k1] == y_true)
        c2 = (model_preds[k2] == y_true)
        b   = int(np.sum( c1 & ~c2))
        c   = int(np.sum(~c1 &  c2))
        tbl = np.array([[int(np.sum(~c1 & ~c2)), b],
                         [c, int(np.sum(c1 & c2))]])
        res = mcnemar(tbl, exact=(b + c < 25))
        mc_records.append({
            'model_1':   k1, 'model_2': k2,
            'statistic': round(float(res.statistic), 4),
            'p_value':   round(float(res.pvalue), 4),
            'sig_p05':   res.pvalue < 0.05,
        })
        print(f'  {k1} vs {k2}: p={res.pvalue:.4f} {"*SIG*" if res.pvalue < 0.05 else ""}')
pd.DataFrame(mc_records).to_csv('results/mcnemar_tests.csv', index=False)

# ── 3. Cohen's Kappa
print('\n3. Cohen Kappa...')
kappa_records_list = []
for mn, preds in model_preds.items():
    k = cohen_kappa_score(y_true, preds)
    kappa_records_list.append({'model': mn, 'is_proposed': mn == PROPOSED_MODEL, 'kappa': round(k, 4)})
    print(f'  {mn}: Kappa={k:.4f}')
pd.DataFrame(kappa_records_list).to_csv('results/cohen_kappa.csv', index=False)

# ── 4. ANOVA + Tukey HSD on XAI Comprehensiveness
print('\n4. ANOVA + Tukey HSD on XAI Comprehensiveness...')
comp_groups = {
    mn: xai_val_df[xai_val_df['model'] == mn]['comprehensiveness'].dropna().values
    for mn in model_preds
}
valid = {k: v for k, v in comp_groups.items() if len(v) >= 2}
if len(valid) >= 2:
    F, p_anova = f_oneway(*valid.values())
    print(f'  ANOVA: F={F:.4f}, p={p_anova:.4f}')
    pd.DataFrame([{'F': round(float(F), 4), 'p_value': round(float(p_anova), 4)}]
                 ).to_csv('results/anova_comprehensiveness.csv', index=False)
    all_v = np.concatenate(list(valid.values()))
    all_l = np.concatenate([[k] * len(v) for k, v in valid.items()])
    tukey = pairwise_tukeyhsd(all_v, all_l, alpha=0.05)
    print(tukey)
    pd.DataFrame(tukey._results_table.data[1:],
                 columns=tukey._results_table.data[0]).to_csv('results/tukey_hsd.csv', index=False)

# ── 5. Wilcoxon: Insertion/Deletion AUC vs random baseline (0.5)
print('\n5. Wilcoxon signed-rank (XAI faithfulness vs random)...')
wilc_records = []
for mn in model_preds:
    sub   = xai_val_df[xai_val_df['model'] == mn]
    ins_v = sub['insertion_auc'].dropna().values
    del_v = sub['deletion_auc'].dropna().values
    row   = {
        'model':    mn,
        'is_proposed': mn == PROPOSED_MODEL,
        'ins_mean': round(float(np.mean(ins_v)), 4) if len(ins_v) else None,
        'del_mean': round(float(np.mean(del_v)), 4) if len(del_v) else None,
    }
    if len(ins_v) >= 6:
        _, p = wilcoxon(ins_v - 0.5)
        row['ins_wilcoxon_p'] = round(float(p), 4)
    if len(del_v) >= 6:
        _, p = wilcoxon(del_v - 0.5)
        row['del_wilcoxon_p'] = round(float(p), 4)
    wilc_records.append(row)
    print(f'  {mn}: ins_auc={row.get("ins_mean")}  del_auc={row.get("del_mean")}')
pd.DataFrame(wilc_records).to_csv('results/wilcoxon_tests.csv', index=False)

# ── 6. Expected Calibration Error (ECE)
print('\n6. Expected Calibration Error (ECE)...')
def ece(y_true, probs, n_bins=10):
    e = 0.0; n = len(y_true)
    for c in range(probs.shape[1]):
        yb = (y_true == c).astype(int); p = probs[:, c]
        for lo, hi in zip(np.linspace(0, 1, n_bins + 1)[:-1],
                          np.linspace(0, 1, n_bins + 1)[1:]):
            mk = (p > lo) & (p <= hi)
            if mk.sum() == 0: continue
            e += mk.sum() / n * abs(yb[mk].mean() - p[mk].mean())
    return float(e / probs.shape[1])

ece_records = []
for mn, probs in model_probs.items():
    v = ece(y_true, probs)
    ece_records.append({'model': mn, 'is_proposed': mn == PROPOSED_MODEL, 'ece': round(v, 4)})
    print(f'  {mn}: ECE={v:.4f}')
pd.DataFrame(ece_records).to_csv('results/ece_calibration.csv', index=False)

# ── 7. Robustness under Gaussian noise
print('\n7. Robustness under Gaussian noise (std=0.10)...')
def add_noise(img, lbl):
    return tf.clip_by_value(img + tf.random.normal(tf.shape(img), stddev=0.10), 0., 1.), lbl
rob_records = []
for mn, model in all_trained_models.items():
    _test_ds_mn = test_ds_propose if mn == PROPOSED_MODEL else test_ds
    test_ds_noisy = _test_ds_mn.map(add_noise)
    acc_orig  = float(accuracy_score(y_true, model_preds[mn]))
    acc_noisy = float(model.evaluate(test_ds_noisy, verbose=0)[1])
    drop      = round((acc_orig - acc_noisy) * 100, 2)
    rob_records.append({
        'model':                    mn,
        'is_proposed':              mn == PROPOSED_MODEL,
        'accuracy_clean (in %)':   round(acc_orig  * 100, 2),
        'accuracy_noisy (in %)':   round(acc_noisy * 100, 2),
        'accuracy_drop (in %)':    drop,
    })
    print(f'  {mn}: clean={acc_orig*100:.2f}%  noisy={acc_noisy*100:.2f}%  drop={drop}%')
pd.DataFrame(rob_records).to_csv('results/robustness.csv', index=False)

# ── 8. McNemar — proposed vs each baseline (dedicated summary)
print(f'\n8. McNemar: Proposed ({PROPOSED_MODEL}) vs each baseline...')
if PROPOSED_MODEL in model_preds:
    prop_preds   = model_preds[PROPOSED_MODEL]
    prop_correct = (prop_preds == y_true)
    prop_acc     = float(np.mean(prop_correct))
    for mn, preds in model_preds.items():
        if mn == PROPOSED_MODEL:
            continue
        base_correct = (preds == y_true)
        b   = int(np.sum( prop_correct & ~base_correct))
        c   = int(np.sum(~prop_correct &  base_correct))
        tbl = np.array([[int(np.sum(~prop_correct & ~base_correct)), b],
                         [c, int(np.sum(prop_correct & base_correct))]])
        res  = mcnemar(tbl, exact=(b + c < 25))
        base_acc = float(np.mean(base_correct))
        if res.pvalue < 0.05 and prop_acc > base_acc:
            outcome = 'proposed BETTER *SIG*'
        elif res.pvalue < 0.05 and prop_acc < base_acc:
            outcome = 'proposed WORSE *SIG*'
        else:
            outcome = 'no significant difference'
        print(f'  {PROPOSED_MODEL} vs {mn}: p={res.pvalue:.4f} — {outcome}')

print('\nAll statistical tests complete.')


# Visualisations

Per-model plots saved to `plots/<model_name>/`:

- **Confusion matrix** — absolute counts, colour-coded
- **ROC curve** — per-class + macro average AUC
- **Precision-Recall curve** — per-class AUC
- **Per-class bar chart** — Precision / Recall / F1
- **Reliability diagram** — calibration per class

Cross-model summary plots saved to `plots/`:

- `proposed_vs_baselines.tiff` — Accuracy / F1 Macro / Cohen Kappa bar chart
- `xai_faithfulness_comparison.tiff` — Comprehensiveness / Insertion / Deletion AUC
- `bootstrap_ci.tiff` — Accuracy with 95% CI error bars
- `robustness_comparison.tiff` — Accuracy drop under Gaussian noise
- `ablation_study.tiff` — Component contribution (accuracy + delta)

In [ ]:
print("\n============================================================")
print("  CELL 37 | Visualisations — all models")
print("============================================================")
from sklearn.metrics import roc_curve
from matplotlib.patches import Patch

y_true = test_labels
y_bin  = label_binarize(y_true, classes=list(range(num_classes)))

for mn, model in trained_models.items():
    preds = model_preds[mn]
    probs = model_probs[mn]
    mdir  = os.path.join('plots', mn)
    os.makedirs(mdir, exist_ok=True)
    label = f'{mn} [PROPOSED]' if mn == PROPOSED_MODEL else mn

    cm = confusion_matrix(y_true, preds)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f'Confusion Matrix — {label}', fontweight='bold')
    ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(os.path.join(mdir, 'confusion_matrix.tiff'), dpi=300)
    plt.close()

    fig, ax = plt.subplots(figsize=(6, 5))
    macro_tpr = np.zeros(100)
    fpr_base  = np.linspace(0, 1, 100)
    for i, cls in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        roc_auc_val = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{cls} (AUC={roc_auc_val:.3f})', linewidth=1.5)
        macro_tpr += np.interp(fpr_base, fpr, tpr)
    macro_tpr /= num_classes
    macro_auc  = auc(fpr_base, macro_tpr)
    ax.plot(fpr_base, macro_tpr, 'k--', linewidth=2,
            label=f'Macro avg (AUC={macro_auc:.3f})')
    ax.plot([0, 1], [0, 1], 'gray', linestyle=':', linewidth=1)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve — {label}', fontweight='bold')
    ax.legend(loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(mdir, 'roc.tiff'), dpi=300)
    plt.close()

    fig, ax = plt.subplots(figsize=(6, 5))
    for i, cls in enumerate(class_names):
        p_, r_, _ = precision_recall_curve(y_bin[:, i], probs[:, i])
        ax.plot(r_, p_, label=f'{cls} (AUC={auc(r_, p_):.3f})', linewidth=1.5)
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'Precision-Recall Curve — {label}', fontweight='bold')
    ax.legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(mdir, 'pr_curve.tiff'), dpi=300)
    plt.close()

    report = classification_report(y_true, preds, target_names=class_names, output_dict=True)
    cls_df = pd.DataFrame({
        'Class':     class_names,
        'Precision': [round(report[c]['precision'], 4) for c in class_names],
        'Recall':    [round(report[c]['recall'],    4) for c in class_names],
        'F1-Score':  [round(report[c]['f1-score'],  4) for c in class_names],
    })
    cls_df.to_csv(os.path.join(mdir, 'per_class_metrics.csv'), index=False)
    x = np.arange(len(class_names)); w = 0.25
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - w, cls_df['Precision'], w, label='Precision', color='#3498db')
    ax.bar(x,     cls_df['Recall'],    w, label='Recall',    color='#2ecc71')
    ax.bar(x + w, cls_df['F1-Score'],  w, label='F1-Score',  color='#e74c3c')
    ax.set_xticks(x); ax.set_xticklabels(class_names, rotation=20, ha='right')
    ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
    ax.set_title(f'Per-Class Metrics — {label}', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(mdir, 'per_class_metrics.tiff'), dpi=300)
    plt.close()

    fig, axes = plt.subplots(1, num_classes, figsize=(4 * num_classes, 4))
    for i, cls in enumerate(class_names):
        ax = axes[i] if num_classes > 1 else axes
        pt, pp = calibration_curve((y_true == i).astype(int), probs[:, i], n_bins=10)
        ax.plot(pp, pt, marker='o', color='#e74c3c', label='Model')
        ax.plot([0, 1], [0, 1], 'k--', label='Perfect')
        ax.set_title(cls, fontsize=9)
        ax.set_xlabel('Mean Predicted Prob')
        ax.set_ylabel('Fraction of Positives')
        ax.legend(fontsize=7)
    fig.suptitle(f'Reliability Diagram — {label}', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(mdir, 'reliability_diagram.tiff'), dpi=300)
    plt.close()

xai_summary = xai_val_df.groupby('model')[
    ['comprehensiveness', 'sufficiency', 'insertion_auc', 'deletion_auc',
     'spearman_rho_gc_saliency']].mean().reset_index()
xai_summary.to_csv('results/xai_faithfulness_summary.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors_xai = ['#e74c3c' if m == PROPOSED_MODEL else '#95a5a6' for m in xai_summary['model']]
for ax, (col, label) in zip(axes, [
    ('comprehensiveness', 'Comprehensiveness'),
    ('insertion_auc',     'Insertion AUC'),
    ('deletion_auc',      'Deletion AUC'),
]):
    ax.bar(xai_summary['model'], xai_summary[col], color=colors_xai, edgecolor='white')
    ax.set_title(label, fontweight='bold')
    ax.set_xticklabels(xai_summary['model'], rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Score')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
axes[0].legend(handles=[
    Patch(facecolor='#e74c3c', label=f'Proposed ({PROPOSED_MODEL})'),
    Patch(facecolor='#95a5a6', label='Baseline'),
], fontsize=8)
fig.suptitle('XAI Faithfulness Metrics — All Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/xai_faithfulness_comparison.tiff', dpi=300)
plt.close()

summary_rows = [
    {'model': mn,
     'accuracy': round(m['accuracy'] * 100, 2),
     'f1_macro': round(m['f1_macro'] * 100, 2),
     'kappa':    round(m['kappa'], 4)}
    for mn, m in all_results.items()
]
sum_df = pd.DataFrame(summary_rows).sort_values('accuracy', ascending=False).reset_index(drop=True)
colors = ['#e74c3c' if r == PROPOSED_MODEL else '#95a5a6' for r in sum_df['model']]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (col, label) in zip(axes, [
    ('accuracy', 'Accuracy (%)'),
    ('f1_macro', 'F1 Macro (%)'),
    ('kappa',    'Cohen Kappa'),
]):
    ax.bar(sum_df['model'], sum_df[col], color=colors, edgecolor='white')
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xticklabels(sum_df['model'], rotation=30, ha='right', fontsize=9)
    ax.set_ylabel(label)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
axes[0].legend(handles=[
    Patch(facecolor='#e74c3c', label=f'Proposed ({PROPOSED_MODEL})'),
    Patch(facecolor='#95a5a6', label='Baseline'),
], fontsize=8)
fig.suptitle('Proposed Model vs Baselines — Performance Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/proposed_vs_baselines.tiff', dpi=300)
plt.close()

boot_plot = boot_df.sort_values('accuracy (in %)', ascending=False).reset_index(drop=True)
colors_b  = ['#e74c3c' if m == PROPOSED_MODEL else '#95a5a6' for m in boot_plot['model']]
yerr_lo   = boot_plot['accuracy (in %)'] - boot_plot['ci_low_95 (in %)']
yerr_hi   = boot_plot['ci_high_95 (in %)'] - boot_plot['accuracy (in %)']
fig, ax   = plt.subplots(figsize=(8, 5))
ax.bar(boot_plot['model'], boot_plot['accuracy (in %)'],
       color=colors_b, edgecolor='white',
       yerr=[yerr_lo, yerr_hi], capsize=5, error_kw={'linewidth': 1.5})
ax.set_ylabel('Accuracy (%)'); ax.set_xlabel('Model')
ax.set_title('Accuracy with 95% Bootstrap CI', fontsize=12, fontweight='bold')
ax.set_xticklabels(boot_plot['model'], rotation=30, ha='right')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('plots/bootstrap_ci.tiff', dpi=300)
plt.close()

rob_df   = pd.read_csv('results/robustness.csv')
colors_r = ['#e74c3c' if m == PROPOSED_MODEL else '#95a5a6' for m in rob_df['model']]
fig, ax  = plt.subplots(figsize=(8, 5))
ax.bar(rob_df['model'], rob_df['accuracy_drop (in %)'], color=colors_r, edgecolor='white')
ax.set_ylabel('Accuracy Drop (%)'); ax.set_xlabel('Model')
ax.set_title('Robustness: Accuracy Drop under Gaussian Noise', fontsize=12, fontweight='bold')
ax.set_xticklabels(rob_df['model'], rotation=30, ha='right')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('plots/robustness_comparison.tiff', dpi=300)
plt.close()

ablation_df = pd.read_csv('results/ablation_study.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_abl = ['#e74c3c' if 'Full' in c else '#95a5a6' for c in ablation_df['Configuration']]
axes[0].barh(ablation_df['Configuration'], ablation_df['Accuracy (in %)'],
             color=colors_abl, edgecolor='white')
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title(f'Ablation Study — {PROPOSED_MODEL} Accuracy', fontweight='bold')
axes[0].axvline(ablation_df['Accuracy (in %)'].iloc[0], color='#e74c3c',
                linestyle='--', linewidth=1.2, label='Full model')
axes[0].legend(fontsize=8)
for i, v in enumerate(ablation_df['Accuracy (in %)']):
    axes[0].text(v + 0.2, i, f'{v:.2f}%', va='center', fontsize=8)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

delta_colors = ['#2ecc71' if d >= 0 else '#e74c3c' for d in ablation_df['Delta vs Full (%)']]
axes[1].barh(ablation_df['Configuration'], ablation_df['Delta vs Full (%)'],
             color=delta_colors, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Delta vs Full Model (%)')
axes[1].set_title('Component Contribution (Delta Accuracy)', fontweight='bold')
for i, v in enumerate(ablation_df['Delta vs Full (%)']):
    axes[1].text(v + (0.1 if v >= 0 else -0.1), i, f'{v:+.2f}%',
                 va='center', ha='left' if v >= 0 else 'right', fontsize=8)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

fig.suptitle(f'Ablation Study — {PROPOSED_MODEL} Component Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/ablation_study.tiff', dpi=300)
plt.close()
print('Saved: plots/ablation_study.tiff')

print('All visualisations saved.')


# Final Export — Consolidated Results

Assembles all per-model metrics into a single ranked table:

**Columns:** model · is_proposed · accuracy · f1_weighted · f1_macro · precision_weighted · recall_weighted · kappa · roc_auc · pr_auc · ci_low_95 · ci_high_95 · comprehensiveness · sufficiency · insertion_auc · deletion_auc · spearman_rho_gc_saliency

Saved to `results/final_consolidated_results.csv`.

Also prints a **proposed vs best baseline** summary showing accuracy margin, F1 margin, and Kappa.

In [ ]:
print("\n============================================================")
print("  CELL 39 | Final export — consolidated results (both regimes)")
print("============================================================")
final_rows = []
for mn in trained_models:
    m    = all_results[mn]
    boot = boot_df[boot_df['model'] == mn].iloc[0]
    krow = next((r for r in kappa_records_list if r['model'] == mn), {})
    sub  = xai_val_df[xai_val_df['model'] == mn]
    xm   = sub[['comprehensiveness', 'sufficiency', 'insertion_auc', 'deletion_auc',
                 'spearman_rho_gc_saliency']].mean().to_dict()
    final_rows.append({
        'model':       mn,
        'is_proposed': mn == PROPOSED_MODEL,
        'accuracy (in %)':           round(m['accuracy']            * 100, 2),
        'f1_weighted (in %)':        round(m['f1_weighted']         * 100, 2),
        'f1_macro (in %)':           round(m['f1_macro']            * 100, 2),
        'precision_weighted (in %)': round(m['precision_weighted']  * 100, 2),
        'recall_weighted (in %)':    round(m['recall_weighted']     * 100, 2),
        'kappa':                     round(krow.get('kappa', 0),  4),
        'roc_auc':                   round(m['roc_auc'],           4) if m.get('roc_auc')  else None,
        'pr_auc':                    round(m['pr_auc'],            4) if m.get('pr_auc')   else None,
        'ci_low_95 (in %)':          float(boot['ci_low_95 (in %)']),
        'ci_high_95 (in %)':         float(boot['ci_high_95 (in %)']),
        'comprehensiveness':         round(xm.get('comprehensiveness',  0), 4),
        'sufficiency':               round(xm.get('sufficiency',        0), 4),
        'insertion_auc':             round(xm.get('insertion_auc',      0), 4),
        'deletion_auc':              round(xm.get('deletion_auc',       0), 4),
        'spearman_rho_gc_saliency':  round(xm.get('spearman_rho_gc_saliency', 0), 4),
    })

final_df = pd.DataFrame(final_rows).sort_values('accuracy (in %)', ascending=False).reset_index(drop=True)
final_df.to_csv('results/final_consolidated_results.csv', index=False)

print('\n=== FINAL CONSOLIDATED RESULTS ===')
print(final_df.to_string(index=False))

proposed_row  = final_df[final_df['is_proposed']]
baseline_rows = final_df[~final_df['is_proposed']]
if not proposed_row.empty and not baseline_rows.empty:
    pr  = proposed_row.iloc[0]
    br  = baseline_rows.iloc[0]   # best baseline (sorted by accuracy)
    print(f'\n=== Proposed model ({PROPOSED_MODEL}) vs best baseline ===')
    print(f'  Proposed : {pr["accuracy (in %)"]:.2f}%  F1m={pr["f1_macro (in %)"]:.2f}%  Kappa={pr["kappa"]:.4f}')
    print(f'  Best base: {br["model"]} — {br["accuracy (in %)"]:.2f}%  F1m={br["f1_macro (in %)"]:.2f}%  Kappa={br["kappa"]:.4f}')
    print(f'  Margin   : acc={pr["accuracy (in %)"]-br["accuracy (in %)"]:.2f}%  f1m={pr["f1_macro (in %)"]-br["f1_macro (in %)"]:.2f}%')

print('\n=== ALL OUTPUT FILES ===')
import glob as _g
for fp in sorted(_g.glob('results/**', recursive=True) +
                 _g.glob('xai_outputs/**', recursive=True) +
                 _g.glob('plots/**', recursive=True)):
    if os.path.isfile(fp):
        print(f'  {fp}  ({round(os.path.getsize(fp)/1024, 1)} KB)')
